# Matched Grokking Setup: Transformer + MLP

This notebook trains two architectures on the same modular addition dataset and hyperparameters:
- single-layer single-head Transformer (Nanda/Poncini style)
- 1-hidden-layer ReLU MLP with separate left/right embeddings (Chughtai style)

The split is 30% train / 70% test by default, using a fixed seed for reproducibility.
The notebook saves checkpoints and training curves for later comparison.

Model classes, dataset construction, the training loop, and checkpoint loading
all live in the `training/` package — this notebook only orchestrates runs.

## Dependencies

Run this cell if you need to install the main Python packages. If your environment already has PyTorch, NumPy, and Matplotlib, you can skip it.

In [ ]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt

from training.config import GrokConfig, set_global_seeds
from training.models import GrokTransformer, ChughtaiMLP
from training.data import make_dataset, pretty_split
from training.loop import train_loop, DEFAULT_RUN_ROOT
from training.checkpoints import load_final_checkpoint

set_global_seeds(0)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

RUN_ROOT = DEFAULT_RUN_ROOT
os.makedirs(RUN_ROOT, exist_ok=True)

In [ ]:
# Load already-trained models from the latest checkpoints under RUN_ROOT.
# Run this cell to skip training and pick up where a previous run left off.

mlp_config, mlp_ckpt, mlp_metrics = load_final_checkpoint('mlp_run', run_root=RUN_ROOT, device=DEVICE)
mlp = ChughtaiMLP(mlp_config.p, d_hidden=mlp_config.mlp_d_hidden).to(DEVICE)
mlp.load_state_dict(mlp_ckpt['model_state'])
mlp.eval()

transformer_config, transformer_ckpt, transformer_metrics = load_final_checkpoint('transformer_run', run_root=RUN_ROOT, device=DEVICE)
transformer = GrokTransformer(transformer_config).to(DEVICE)
transformer.load_state_dict(transformer_ckpt['model_state'])
transformer.eval()

print('Both models loaded.')

In [ ]:
config = GrokConfig()
train_data, train_labels, test_data, test_labels = make_dataset(config, device=DEVICE)
pretty_split(train_data, test_data)
print('Transformer vocabulary size:', config.d_vocab)
print('First train example:', train_data[0].tolist(), 'label', train_labels[0].item())

In [ ]:
transformer = GrokTransformer(config).to(DEVICE)
print('Transformer parameter count:', sum(p.numel() for p in transformer.parameters()))
transformer_metrics = train_loop(
    transformer, train_data, train_labels, test_data, test_labels,
    config, num_epochs=config.transformer_epochs,
    run_name='transformer_run', is_transformer=True, run_root=RUN_ROOT,
)

In [ ]:
mlp = ChughtaiMLP(config.p, d_hidden=config.mlp_d_hidden).to(DEVICE)
print('MLP parameter count:', sum(p.numel() for p in mlp.parameters()))
mlp_metrics = train_loop(
    mlp, train_data, train_labels, test_data, test_labels,
    config, num_epochs=config.mlp_epochs,
    run_name='mlp_run', is_transformer=False, run_root=RUN_ROOT,
)

In [ ]:
def plot_training(metrics, title):
    epochs = np.arange(len(metrics['train_loss'])) + 1
    fig, axs = plt.subplots(1, 2, figsize=(14, 5))

    axs[0].plot(epochs, metrics['train_loss'], label='train')
    axs[0].plot(epochs, metrics['test_loss'], label='test')
    axs[0].set_xscale('log')
    axs[0].set_yscale('log')
    axs[0].set_xlabel('Epoch')
    axs[0].set_ylabel('Cross-entropy loss')
    axs[0].set_title(f'{title} loss')
    axs[0].legend()

    axs[1].plot(epochs, metrics['train_acc'], label='train')
    axs[1].plot(epochs, metrics['test_acc'], label='test')
    axs[1].set_xscale('log')
    axs[1].set_xlabel('Epoch')
    axs[1].set_ylabel('Accuracy')
    axs[1].set_title(f'{title} accuracy')
    axs[1].axhline(0.99, ls='--', alpha=0.4, color='gray', label='99%')
    axs[1].legend()

    plt.tight_layout()
    plt.show()


plot_training(transformer_metrics, 'Transformer')
plot_training(mlp_metrics, 'MLP')
print('Transformer final test accuracy:', transformer_metrics['test_acc'][-1])
print('MLP final test accuracy:', mlp_metrics['test_acc'][-1])